# Understand Tool Runner, one cell at a time

You are helping a customer whose hoodie is late. They want a return **only if no replacement is available**. Start with one lookup, expose it as a tool, then let Tool Runner carry the request through to an answer.

**Practice mode:** the model replies are scripted; Anthropic's actual SDK runs real Python tools and saves real SQLite records. No Claude API key is needed and there are no model charges. This notebook does not call Claude.

Run the setup once, then run Part 1's five cells in order. Part 2 has five deeper experiments, each with two versions to compare. All lesson cells are identical to the lab UI. In Colab, you can edit them and experiment. If you change an earlier cell, rerun the cells below it.

`support_session(stock=...)` supplies a fresh practice store, its order/stock/return tools, and an offline SDK client. It does not run the Tool Runner for you. Expand setup to inspect those definitions.

In [ ]:
# @title Prepare the practice store and offline SDK client
# This setup runs once. Expand it to inspect the store and scripted replies.
%pip -q install anthropic==1.12.1 httpx2==2.13.1

import atexit
import contextlib
import json
import types
import anthropic
import httpx2 as httpx
from anthropic import beta_tool
from anthropic.lib.tools import ToolError

MODEL = 'claude-haiku-5-5'
CUSTOMER_REQUEST = "What's the status of order A100? Check replacement stock for Blue hoodie (M). If none is available, please start a return because the delivery is too late."
lab = types.ModuleType('practice_store')
exec('from __future__ import annotations\nimport json\nimport sqlite3\nfrom typing import Any\nimport anthropic\nfrom anthropic.lib.tools import ToolError\n\ndef _error(code: str, message: str) -> ToolError:\n    return ToolError(json.dumps({"error": code, "message": message}))\n\n\ndef _order_id(value: str) -> str:\n    if not isinstance(value, str) or not value.strip() or len(value) > 100:\n        raise _error("invalid_order_id", "Provide a non-empty order ID shorter than 100 characters.")\n    return value.strip().upper()\n\n\ndef _product_key(value: str) -> str:\n    if not isinstance(value, str) or not value.strip() or len(value) > 200:\n        raise _error("invalid_product", "Provide a non-empty product name shorter than 200 characters.")\n    return " ".join(value.strip().casefold().split())\n\n\nclass Store:\n    """Each run owns a fresh in-memory SQLite database, never shared with others."""\n\n    def __init__(self, inventory_count: int = 0, inventory_outage: bool = False, simulate_missing_write: bool = False):\n        self.db = sqlite3.connect(":memory:")\n        self.db.row_factory = sqlite3.Row\n        self.inventory_outage = inventory_outage\n        self.simulate_missing_write = simulate_missing_write\n        self.db.executescript("""\n            PRAGMA foreign_keys=ON;\n            CREATE TABLE orders (order_id TEXT PRIMARY KEY, product TEXT NOT NULL,\n                status TEXT NOT NULL, eta TEXT NOT NULL, returnable INTEGER NOT NULL);\n            CREATE TABLE inventory (product TEXT PRIMARY KEY, product_key TEXT UNIQUE NOT NULL,\n                count INTEGER NOT NULL CHECK(count >= 0));\n            CREATE TABLE returns (return_id TEXT PRIMARY KEY, order_id TEXT UNIQUE NOT NULL\n                REFERENCES orders(order_id), reason TEXT NOT NULL);\n        """)\n        self.db.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", [\n            ("A100", "Blue hoodie (M)", "shipped", "Friday", 1),\n            ("A101", "Trail shoes (10)", "delivered", "Monday", 1),\n            ("A102", "Gift card", "delivered", "Monday", 0),\n        ])\n        self.db.executemany("INSERT INTO inventory VALUES (?, ?, ?)", [\n            (product, _product_key(product), count) for product, count in [\n                ("Blue hoodie (M)", inventory_count), ("Trail shoes (10)", 12), ("Gift card", 999)\n            ]\n        ])\n        self.db.commit()\n\n    def close(self) -> None:\n        self.db.close()\n\n    def snapshot(self) -> dict:\n        orders = [dict(row) for row in self.db.execute("SELECT * FROM orders ORDER BY order_id")]\n        for row in orders:\n            row["returnable"] = bool(row["returnable"])\n        return {"orders": orders,\n                "inventory": [dict(row) for row in self.db.execute("SELECT product, count FROM inventory ORDER BY product")],\n                "returns": [dict(row) for row in self.db.execute("SELECT * FROM returns ORDER BY order_id")]}\n\n    def lookup_order(self, order_id: str) -> dict:\n        """Get an existing order by its normalized identifier."""\n        normalized = _order_id(order_id)\n        row = self.db.execute("SELECT * FROM orders WHERE order_id = ?", (normalized,)).fetchone()\n        if row is None:\n            raise _error("order_not_found", "No order matches the supplied identifier.")\n        result = dict(row)\n        result["returnable"] = bool(result["returnable"])\n        return result\n\n    def check_inventory(self, product: str) -> dict:\n        """Look up replacement stock, preserving a valid zero count."""\n        normalized = _product_key(product)\n        if self.inventory_outage:\n            raise _error("inventory_unavailable", "Inventory service is temporarily unavailable. Stock is unknown.")\n        row = self.db.execute("SELECT product, count FROM inventory WHERE product_key = ?", (normalized,)).fetchone()\n        if row is None:\n            raise _error("product_not_found", "No inventory record matches the supplied product.")\n        return dict(row)\n\n    def check_return_eligibility(self, order_id: str) -> dict:\n        """Inspect policy without creating a return."""\n        order = self.lookup_order(order_id)\n        return {"order_id": order["order_id"], "eligible": order["returnable"],\n                "reason": "This order is returnable." if order["returnable"] else "Gift cards are non-returnable."}\n\n    def start_return(self, order_id: str, reason: str) -> dict:\n        """Enforce policy and persist an idempotent return record."""\n        order = self.lookup_order(order_id)\n        if not order["returnable"]:\n            raise _error("return_not_allowed", "This order is non-returnable. No return was created.")\n        if not isinstance(reason, str) or not reason.strip() or len(reason) > 1000:\n            raise _error("invalid_reason", "Provide a non-empty return reason of at most 1000 characters.")\n        order_id = order["order_id"]\n        previous = self.db.execute("SELECT * FROM returns WHERE order_id = ?", (order_id,)).fetchone()\n        if previous is not None:\n            return {**dict(previous), "already_exists": True}\n        record = {"return_id": "R-" + order_id, "order_id": order_id, "reason": reason.strip()}\n        if not self.simulate_missing_write:\n            # The UNIQUE constraint remains authoritative even if callers retry.\n            self.db.execute("INSERT INTO returns (return_id, order_id, reason) VALUES (?, ?, ?) ON CONFLICT(order_id) DO NOTHING",\n                            (record["return_id"], order_id, record["reason"]))\n            self.db.commit()\n        return {**record, "already_exists": False}\n\n\ndef _build_tools(store: Store, snapshots: list[dict]) -> list:\n    """The SDK generates JSON schemas from these type hints and docstrings."""\n    def invoke(name: str, **kwargs) -> str:\n        try:\n            return json.dumps(getattr(store, name)(**kwargs))\n        finally:\n            snapshots.append({"name": name, "state": store.snapshot()})\n\n    @anthropic.beta_tool\n    def lookup_order(order_id: str) -> str:\n        """Look up a customer order. Returns product, delivery status, ETA, and returnability.\n        Use this to obtain facts before answering an order question. Unknown IDs produce an error.\n\n        Args:\n            order_id: Order identifier, for example A100. Whitespace and case are normalized.\n        """\n        return invoke("lookup_order", order_id=order_id)\n\n    @anthropic.beta_tool\n    def check_inventory(product: str) -> str:\n        """Check current replacement stock. Returns product and count, including zero.\n        Unknown products and service outages produce errors. Never treat an error as stock availability.\n\n        Args:\n            product: Product name from an order, such as Blue hoodie (M), Trail shoes (10), or Gift card.\n        """\n        return invoke("check_inventory", product=product)\n\n    @anthropic.beta_tool\n    def start_return(order_id: str, reason: str) -> str:\n        """Create a return for an existing returnable order when the customer requests it.\n        Enforces policy in Python. Repeated requests reuse the same return record. Returns\n        return_id, order_id, reason, and already_exists. Eligibility questions alone do not authorize a return.\n\n        Args:\n            order_id: Existing order ID to return.\n            reason: A short reason supplied by the customer.\n        """\n        return invoke("start_return", order_id=order_id, reason=reason)\n\n    @anthropic.beta_tool\n    def check_return_eligibility(order_id: str) -> str:\n        """Check whether an existing order can be returned without changing any application state.\n        Returns eligible and an explanation. Use for questions about whether an order can be returned;\n        such a question does not authorize starting a return. Unknown orders produce an error.\n\n        Args:\n            order_id: Order ID to inspect.\n        """\n        return invoke("check_return_eligibility", order_id=order_id)\n\n    return [lookup_order, check_inventory, start_return, check_return_eligibility]\n\n\ndef _decode_content(content: Any) -> Any:\n    if isinstance(content, list):\n        content = "\\n".join(str(item.get("text", "")) for item in content if isinstance(item, dict))\n    if isinstance(content, str):\n        try:\n            return json.loads(content)\n        except (ValueError, TypeError):\n            return content[:4000]\n    return content\n\n\ndef _fixture_response(scenario: str, turn: int, body: dict, model: str) -> dict:\n    """Script only model messages. All tool results originate in the actual runner."""\n    previous = [block for message in body.get("messages", []) if message.get("role") == "user"\n                for block in (message.get("content") if isinstance(message.get("content"), list) else [])\n                if block.get("type") == "tool_result"]\n    values = [_decode_content(block.get("content")) for block in previous]\n    successful_returns = [value for value in values if isinstance(value, dict) and "return_id" in value]\n    calls: list[tuple[str, dict]] = []\n    answer = ""\n    if scenario == "normal":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": " a100 "}), ("check_inventory", {"product": " blue HOODIE (m) "})]\n        elif turn == 1:\n            stock = next((value for value in reversed(values) if isinstance(value, dict) and "count" in value), None)\n            if stock is None:\n                answer = "Order A100 is shipped, with an ETA of Friday. Inventory could not be checked, so I have not assumed stock or started a return."\n            elif stock["count"] == 0:\n                calls = [("start_return", {"order_id": "A100", "reason": "Delivery is too late"})]\n            else:\n                answer = f"Order A100 is shipped, with an ETA of Friday. There are {stock[\'count\']} replacement hoodies available. I have not started a return because your condition was that replacements be out of stock."\n        else:\n            answer = "Order A100 is shipped and due Friday. No replacement hoodie is in stock. The return tool reports return R-A100 created for your order." if successful_returns else "The return could not be started. Please review the tool result."\n    elif scenario == "unknown":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": "A999"})]\n        else:\n            answer = "I could not find order A999. No return was created. Please check the order ID."\n    elif scenario == "policy":\n        if turn == 0:\n            calls = [("lookup_order", {"order_id": "A102"})]\n        elif turn == 1:\n            calls = [("start_return", {"order_id": "A102", "reason": "Customer requested an exception"})]\n        else:\n            answer = "Order A102 is a non-returnable gift card. The return tool rejected the request, and no return was created."\n    elif scenario in ("outage", "batch_errors"):\n        names = ["Blue hoodie (M)", "Trail shoes (10)", "Gift card"] if scenario == "batch_errors" else ["Blue hoodie (M)"]\n        calls = [("check_inventory", {"product": name}) for name in names]\n    elif scenario == "duplicate":\n        if turn == 0:\n            calls = [("start_return", {"order_id": order_id, "reason": "Shoes do not fit"}) for order_id in ("A101", " a101 ")]\n        else:\n            answer = "Both calls returned R-A101. The second call found the existing return, so there is one return for order A101." if any(isinstance(value, dict) and value.get("already_exists") for value in values) else "The return tool reported a return, but the second response did not report an existing record. Review the state verification."\n    elif scenario == "eligibility":\n        if turn == 0:\n            calls = [("check_return_eligibility", {"order_id": "A101"})]\n        else:\n            eligible = next((value.get("eligible") for value in reversed(values) if isinstance(value, dict) and "eligible" in value), False)\n            answer = "Order A101 is eligible for a return. I only checked eligibility; no return has been started." if eligible else "The order is not eligible for a return. No return was started."\n    content = ([{"type": "tool_use", "id": f"toolu_fixture_{turn}_{index}", "name": name, "input": args}\n                for index, (name, args) in enumerate(calls)] if calls else [{"type": "text", "text": answer}])\n    return {"id": f"msg_fixture_{turn}", "type": "message", "role": "assistant", "model": model,\n            "content": content, "stop_reason": "tool_use" if calls else "end_turn", "stop_sequence": None,\n            "usage": {"input_tokens": 0, "output_tokens": 0}}\n', lab.__dict__)

class SupportSession:
    """One isolated store, its tools, and an SDK client with an offline transport."""

    def __init__(self, stock=0, inventory_outage=False, simulate_missing_write=False):
        self.store = lab.Store(inventory_count=stock, inventory_outage=inventory_outage,
                               simulate_missing_write=simulate_missing_write)
        self.calls = []
        self.requests = []
        self.tools = {tool.to_dict()["name"]: tool
                      for tool in lab._build_tools(self.store, [])
                      if tool.to_dict()["name"] != "check_return_eligibility"}

        def respond(request):
            body = json.loads(request.content)
            self.requests.append(body)
            # Derive the turn from this conversation, so a learner may rerun a
            # cell without the fixture's turn counter leaking across runners.
            turn = sum(message["role"] == "assistant" for message in body["messages"])
            reply = lab._fixture_response("normal", turn, body, MODEL)
            self.calls.extend(block["name"] for block in reply["content"] if block["type"] == "tool_use")
            return httpx.Response(200, json=reply)

        self.http = httpx.Client(transport=httpx.MockTransport(respond))
        try:
            self.client = anthropic.Anthropic(
                api_key="fixture-key-not-a-real-credential", base_url="https://fixture.invalid",
                http_client=self.http, max_retries=0,
            )
        except Exception:
            self.http.close()
            self.store.close()
            raise

    def close(self):
        try:
            self.client.close()
        finally:
            self.store.close()


if 'resources' in globals():
    resources.close()
resources = contextlib.ExitStack()
atexit.register(resources.close)

def support_session(stock=0, **options):
    session = SupportSession(stock, **options)
    resources.callback(session.close)
    return session

print('Scripted model replies. Real Tool Runner and Python tools. No API key or model charge.')
print('Ready. Run Part 1, then try the experiments in Part 2.')

**Customer request**

> What's the status of order A100? Check replacement stock for Blue hoodie (M). If none is available, please start a return because the delivery is too late.

## 1. Read one fact

The setup supplies the same practice store as the bot. Run its inventory lookup directly. What does a count of zero tell you?

In [ ]:
baseline = support_session(stock=0)
baseline.store.check_inventory("Blue hoodie (M)")

**What to notice:** Python reads the store. Zero is a real stock count, not a guess from the model.

## 2. Make it a tool

Wrap the lookup in a typed function. Run it to see the input description the SDK generates for the model.

In [ ]:
@beta_tool
def check_inventory(product: str) -> str:
    """Check replacement stock for a product."""
    return json.dumps(baseline.store.check_inventory(product))

baseline.tools["check_inventory"] = check_inventory
check_inventory.to_dict()["input_schema"]

**What to notice:** @beta_tool describes the Python function so the model can request it. The product argument becomes a required string.

## 3. Let Tool Runner handle the loop

Run the customer's request with the tools attached. until_done() continues through tool calls until the model finishes.

In [ ]:
runner = baseline.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512,
    tools=list(baseline.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
answer = runner.until_done()
print(answer.content[0].text)

**What to notice:** Tool Runner dispatches requested tools, sends their results back, and continues the conversation. You supplied the tools; the SDK handled the loop.

## 4. Change one fact

Predict: will a return still be created with three replacements available? Run the same request and tools against a fresh store with stock=3.

In [ ]:
comparison = support_session(stock=3)
runner = comparison.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512,
    tools=list(comparison.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
print(runner.until_done().content[0].text)

**What to notice:** The request is unchanged. A different tool result changes the scripted model's next action: available stock means no return.

## 5. Check what actually changed

A reply is not proof that an action happened. Run these assertions against both databases to check the result.

In [ ]:
before = baseline.store.snapshot()["returns"]
after = comparison.store.snapshot()["returns"]
assert [row["order_id"] for row in before] == ["A100"]
assert after == []
print("Checks passed: 0 in stock → 1 saved return; 3 in stock → 0 returns.")

**What to notice:** Both database checks passed. Tool Runner coordinates the loop; your Python tools and checks own the real application behavior.

## You have completed the loop

Python owns the facts and actions. `@beta_tool` describes what the model may request. Tool Runner dispatches those requests and returns tool results until the conversation finishes. Your assertions check the saved outcome.

Try a new stock count in cell 4. Before running it, predict whether a return should exist. The fixed model replies respond to stock changes; arbitrary new prompts need a live model.

[Read the Tool Runner documentation](https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner).

# Part 2: Why Tool Runner, and how to control it

Each experiment changes one condition. **Predict → run → change → check.** Run both versions before moving on. Each version creates its own fresh store, so these cells can be rerun independently.

The model is still scripted. These experiments show what the installed SDK does, while making our model behavior and application policy visible. Tool Runner is a beta SDK helper; this notebook pins the version used by the lab.

## Experiment 1: Why use Tool Runner?

A model can request a tool, but your program must execute it and send the result back. Run a minimal manual loop, then let Tool Runner do the same conversation bookkeeping. Both use the same model replies, Python tools, and store.

**Predict:** which bookkeeping does the SDK take over?

### Run: Your own loop

In [ ]:
session = support_session(stock=0)
messages = [{"role": "user", "content": CUSTOMER_REQUEST}]
for turn in range(3):
    # Ask the model, carrying the conversation so far.
    message = session.client.beta.messages.create(
        model=MODEL, max_tokens=512, messages=messages,
        tools=[tool.to_dict() for tool in session.tools.values()],
    )
    messages.append({"role": "assistant", "content": [
        block.model_dump(exclude_none=True) for block in message.content]})
    if message.stop_reason == "end_turn":
        break
    if message.stop_reason != "tool_use":
        raise RuntimeError("The model stopped without completing the task.")
    # Dispatch every requested tool yourself.
    results = []
    for block in message.content:
        if block.type != "tool_use":
            continue
        result = {"type": "tool_result", "tool_use_id": block.id}
        try:
            result["content"] = session.tools[block.name].call(block.input)
        except ToolError as error:
            result.update(content=str(error), is_error=True)
        results.append(result)
    # Send results with matching call IDs; the next turn uses them.
    messages.append({"role": "user", "content": results})
else:
    raise RuntimeError("Reached the turn limit before a final answer.")
saved = session.store.snapshot()["returns"]
assert [row["order_id"] for row in saved] == ["A100"]
print(message.content[0].text)
print("Model turns:", len(session.requests), "| Saved returns:", len(saved))

**Check:** You wrote the conversation bookkeeping: request messages, execute tools, match result IDs, and repeat. This minimal teaching loop handles ToolError and a turn cap; a production loop needs further handling for your application's requirements.

### Change and rerun: Tool Runner

In [ ]:
session = support_session(stock=0)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    max_iterations=3,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
message = runner.until_done()
saved = session.store.snapshot()["returns"]
assert message.stop_reason == "end_turn"
assert [row["order_id"] for row in saved] == ["A100"]
print(message.content[0].text)
print("Model turns:", len(session.requests), "| Saved returns:", len(saved))

**Check:** The SDK replaces that loop for standard tool use, with the same tools and outcome. You still own the business logic and checks. Write a manual loop when your application's needs do not fit the runner's controls.

## Experiment 2: Watch each turn

Run the same request with two stock counts. Each model turn is followed by real tool results. Calling generate_tool_call_response() here lets you inspect the results; the SDK caches them so they are not executed twice.

**Predict:** which tool call disappears when replacements are available?

### Run: 0 in stock

In [ ]:
session = support_session(stock=0)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
for turn, message in enumerate(runner, 1):
    print(f"Turn {turn}: {message.stop_reason}")
    names = {block.id: block.name for block in message.content
             if block.type == "tool_use"}
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            print(" ", names[result["tool_use_id"]], "→", result["content"])
    else:
        print(message.content[0].text)
saved = session.store.snapshot()["returns"]
print(f"Saved returns: {len(saved)}")
assert len(saved) == 1

**Check:** Three model turns: lookup and inventory, then create the return, then answer. Two tools can run within one turn.

### Change and rerun: 3 in stock

In [ ]:
session = support_session(stock=3)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
for turn, message in enumerate(runner, 1):
    print(f"Turn {turn}: {message.stop_reason}")
    names = {block.id: block.name for block in message.content
             if block.type == "tool_use"}
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            print(" ", names[result["tool_use_id"]], "→", result["content"])
    else:
        print(message.content[0].text)
saved = session.store.snapshot()["returns"]
print(f"Saved returns: {len(saved)}")
assert len(saved) == 0

**Check:** The stock result removes the need for start_return. The scripted model finishes after two turns; the database stays unchanged.

## Experiment 3: Stop before the answer

Change max_iterations from two to three. The limit counts model turns, not individual tool calls. Inspect both the last stop reason and the database.

**Predict:** with only two model turns, will the return already exist?

### Run: 2 turns

In [ ]:
session = support_session(stock=0)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    max_iterations=2,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
last_message = runner.until_done()
saved = session.store.snapshot()["returns"]
print(f"Last stop reason: {last_message.stop_reason}")
print(f"Saved returns: {len(saved)}")
if last_message.stop_reason == "end_turn":
    print(last_message.content[0].text)
else:
    print("Stopped before a final answer. Check the saved returns above.")
assert last_message.stop_reason == 'tool_use'
assert len(saved) == 1

**Check:** The return was saved, but the cap stopped the loop before the final answer. until_done() returns the last message, even when its stop reason is still tool_use.

### Change and rerun: 3 turns

In [ ]:
session = support_session(stock=0)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    max_iterations=3,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
last_message = runner.until_done()
saved = session.store.snapshot()["returns"]
print(f"Last stop reason: {last_message.stop_reason}")
print(f"Saved returns: {len(saved)}")
if last_message.stop_reason == "end_turn":
    print(last_message.content[0].text)
else:
    print("Stopped before a final answer. Check the saved returns above.")
assert last_message.stop_reason == 'end_turn'
assert len(saved) == 1

**Check:** The third model turn produces the answer. A turn limit bounds work; it does not roll back actions already taken or prove the task completed.

## Experiment 4: Handle a tool failure

The inventory tool raises ToolError during an outage. Inspect how the SDK turns that exception into a tool result, then restore the service and rerun the same request.

**Predict:** does an inventory failure mean there are zero replacements?

### Run: Unavailable

In [ ]:
session = support_session(stock=0, inventory_outage=True)
# During an outage, check_inventory raises ToolError.
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
inventory_result = None
for message in runner:
    names = {block.id: block.name for block in message.content
             if block.type == "tool_use"}
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            if names[result["tool_use_id"]] == "check_inventory":
                inventory_result = result
print("is_error:", inventory_result.get("is_error", False))
print(inventory_result["content"])
print(message.content[0].text)
saved = session.store.snapshot()["returns"]
print(f"Saved returns: {len(saved)}")
assert inventory_result.get("is_error", False) is True
assert len(saved) == 0

**Check:** is_error=True means stock is unknown. The SDK reports the error to the model; our scripted model chooses to stop without a return. Tool Runner does not decide that business policy.

### Change and rerun: Restored

In [ ]:
session = support_session(stock=0, inventory_outage=False)
# During an outage, check_inventory raises ToolError.
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
inventory_result = None
for message in runner:
    names = {block.id: block.name for block in message.content
             if block.type == "tool_use"}
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            if names[result["tool_use_id"]] == "check_inventory":
                inventory_result = result
print("is_error:", inventory_result.get("is_error", False))
print(inventory_result["content"])
print(message.content[0].text)
saved = session.store.snapshot()["returns"]
print(f"Saved returns: {len(saved)}")
assert inventory_result.get("is_error", False) is False
assert len(saved) == 1

**Check:** The lookup now returns a valid count of zero. The customer's condition is met, and the return is saved. A known zero and an unknown value require different decisions.

## Experiment 5: Catch a convincing mistake

Inject a tool bug that reports success but skips the database write. Compare the claimed return ID with saved records, then fix the write and repeat.

**Predict:** can Tool Runner finish normally even though the return was never saved?

### Run: Skip the write

In [ ]:
session = support_session(stock=0, simulate_missing_write=True)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
for message in runner:
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            value = json.loads(result["content"])
            if "return_id" in value:
                claimed_id = value["return_id"]
saved = session.store.snapshot()["returns"]
verified = any(row["return_id"] == claimed_id for row in saved)
print(message.content[0].text)
print(f"Tool claimed: {claimed_id} | Saved returns: {len(saved)}")
print("Verified: the claimed return is saved." if verified else
      f"Bug caught: reply claims {claimed_id}, database has 0 returns.")
assert verified is False

**Check:** Bug caught: the tool and reply claim success, but there is no saved return. A completed loop cannot verify that your tool implemented its promise.

### Change and rerun: Save the return

In [ ]:
session = support_session(stock=0, simulate_missing_write=False)
runner = session.client.beta.messages.tool_runner(
    model=MODEL, max_tokens=512, run_tools_eagerly=False,
    tools=list(session.tools.values()),
    messages=[{"role": "user", "content": CUSTOMER_REQUEST}],
)
for message in runner:
    response = runner.generate_tool_call_response()
    if response:
        for result in response["content"]:
            value = json.loads(result["content"])
            if "return_id" in value:
                claimed_id = value["return_id"]
saved = session.store.snapshot()["returns"]
verified = any(row["return_id"] == claimed_id for row in saved)
print(message.content[0].text)
print(f"Tool claimed: {claimed_id} | Saved returns: {len(saved)}")
print("Verified: the claimed return is saved." if verified else
      f"Bug caught: reply claims {claimed_id}, database has 0 returns.")
assert verified is True

**Check:** The claimed ID now matches a real record. Tool Runner owns orchestration; your tool implementation and outcome checks own correctness.

## What you now know

You can inspect each model turn, bound the loop, distinguish tool failures from valid data, and verify a reported action against saved state. Tool Runner handles tool dispatch and conversation continuation. Your application supplies policy, reliable tools, and outcome checks.

**Try your own test:** change the cap to one turn. Predict which tools will run, the last stop reason, and whether a return will exist. Before running, update the expected assertions to test your prediction. Then run it and inspect the saved state.

[Tool Runner reference](https://platform.claude.com/docs/en/agents-and-tools/tool-use/tool-runner)

In [ ]:
# @title Close the practice stores when you are finished
resources.close()
print('Practice stores closed. Rerun setup to start again.')